# Notebook - Modelo Linear
## 1. Importação das bibliotecas
Bibliotecas pandas importada para trabalhar com dados e statsmodel para gerar modelo.

In [1]:
import pandas as pd
!pip install statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

Defaulting to user installation because normal site-packages is not writeable


## 2. Importar Dados

In [2]:
all_data = pd.read_csv("../results/10000000.csv")
all_data.head()

,vector_type,algorithm,size,time_ms,stress,carga,ram_kb_max,ram_kb_media,energy_j,comparisons,swaps
0,endnoise,HeapSort,10000000,703.818,none,0.0,43112,42886,12.84420,446056168,232185536
1,endnoise,MergeSort,10000000,508.425,none,0.0,81968,49241,8.68302,119525953,0
2,endnoise,QuickSort,10000000,368.159,none,0.0,43132,42938,8.07957,530500174,161104803
3,ordered,HeapSort,10000000,617.532,none,0.0,43128,42934,14.37630,445162673,231881708
4,ordered,MergeSort,10000000,501.121,none,0.0,81968,48527,9.24516,118788160,0


## 3. Converter valores para categóricosa para construção do modelo


In [3]:
all_data["algorithm"] = all_data["algorithm"].astype("category")
all_data["vector_type"] = all_data["vector_type"].astype("category")
all_data["stress"] = all_data["stress"].astype("category")
all_data["carga"] = all_data["carga"].astype("category")

## 4. Ajustar o modelo linear
Utilizar as 4 categorias + as 2 mais importantes (algoritmoXestresse e algortimoXtipovetor)

In [4]:
modelo = smf.ols(
    """
    time_ms ~
    C(algorithm)
    + C(vector_type)
    + C(stress)
    + C(carga)
    + C(algorithm):C(stress)
    + C(vector_type):C(algorithm)
    """,
    data=all_data
).fit()

## 5. Gerar a ANOVA

In [5]:
anova = anova_lm(modelo, typ=2)

anova

,sum_sq,df,F,PR(>F)
C(algorithm),3.214121e+07,2.0,165.588739,5.678544e-61
C(vector_type),1.664813e+08,7.0,245.056378,9.359004e-195
C(stress),4.123503e+07,3.0,141.626186,1.051589e-73
C(carga),8.779272e+07,2.0,452.300458,1.968319e-132
C(algorithm):C(stress),1.070635e+06,6.0,1.838606,8.887342e-02
C(vector_type):C(algorithm),4.342379e+08,14.0,319.593722,0.000000e+00
Residual,7.822337e+07,806.0,NaN,NaN


## 6. Tabela pronta para o relatório

In [6]:
tabela = (
    anova[["F", "PR(>F)"]]
    .rename(columns={
        "F": "F",
        "PR(>F)": "p"
    })
)

display(tabela)

,F,p
C(algorithm),165.588739,5.678544e-61
C(vector_type),245.056378,9.359004e-195
C(stress),141.626186,1.051589e-73
C(carga),452.300458,1.968319e-132
C(algorithm):C(stress),1.838606,8.887342e-02
C(vector_type):C(algorithm),319.593722,0.000000e+00
Residual,NaN,NaN


## 7. Formatação ACM

In [7]:
tabela = tabela.copy()

tabela["F"] = tabela["F"].round(2)

tabela["p"] = tabela["p"].apply(
    lambda x: "<0.001" if x < 0.001 else f"{x:.3f}"
)

display(tabela)

,F,p
C(algorithm),165.59,<0.001
C(vector_type),245.06,<0.001
C(stress),141.63,<0.001
C(carga),452.30,<0.001
C(algorithm):C(stress),1.84,0.089
C(vector_type):C(algorithm),319.59,<0.001
Residual,NaN,nan


A análise de variância (ANOVA) revelou que o algoritmo utilizado, o tipo de vetor, o nível de estresse e a intensidade da carga exercem influência estatisticamente significativa sobre o tempo de execução (p < 0,001). Além disso, observou-se uma interação significativa entre algoritmo e tipo de vetor (F = 319,59; p < 0,001), indicando que o desempenho relativo dos algoritmos depende das características da entrada. Em contraste, a interação entre algoritmo e estresse não foi estatisticamente significativa (F = 1,84; p = 0,089), sugerindo que os diferentes algoritmos apresentam comportamento semelhante frente aos cenários de contenção de recursos, ainda que o estresse aumente o tempo absoluto de execução.

## Eta quadrado parcial (η² parcial)

In [9]:
anova_eta = anova.copy()

ss_error = anova_eta.loc["Residual","sum_sq"]

anova_eta["eta2_parcial"] = (
    anova_eta["sum_sq"] /
    (anova_eta["sum_sq"] + ss_error)
)

display(anova_eta[["F","PR(>F)","eta2_parcial"]])

,F,PR(>F),eta2_parcial
C(algorithm),165.588739,5.678544e-61,0.291228
C(vector_type),245.056378,9.359004e-195,0.680336
C(stress),141.626186,1.051589e-73,0.345183
C(carga),452.300458,1.968319e-132,0.528821
C(algorithm):C(stress),1.838606,8.887342e-02,0.013502
C(vector_type):C(algorithm),319.593722,0.000000e+00,0.847358
Residual,NaN,NaN,0.500000


## Formatação para artigo

In [11]:
resultado = anova_eta[["F","PR(>F)","eta2_parcial"]].copy()

resultado.rename(columns={
    "PR(>F)": "p",
    "eta2_parcial": "η² parcial"
}, inplace=True)

resultado["F"] = resultado["F"].round(2)

resultado["η² parcial"] = resultado["η² parcial"].round(3)

resultado["p"] = resultado["p"].apply(
    lambda x: "<0.001" if x < 0.001 else f"{x:.3f}"
)
resultado = resultado.drop(index="Residual")
display(resultado)

,F,p,η² parcial
C(algorithm),165.59,<0.001,0.291
C(vector_type),245.06,<0.001,0.680
C(stress),141.63,<0.001,0.345
C(carga),452.30,<0.001,0.529
C(algorithm):C(stress),1.84,0.089,0.014
C(vector_type):C(algorithm),319.59,<0.001,0.847


A ANOVA indicou efeitos significativos do algoritmo (F = 165,59; p < 0,001; η²p = 0,291), do tipo de vetor (F = 245,06; p < 0,001; η²p = 0,680), do tipo de estresse (F = 141,63; p < 0,001; η²p = 0,345) e da intensidade da carga (F = 452,30; p < 0,001; η²p = 0,529) sobre o tempo de execução. Observou-se ainda uma interação altamente significativa entre algoritmo e tipo de vetor (F = 319,59; p < 0,001; η²p = 0,847), evidenciando que o desempenho relativo dos algoritmos depende fortemente das características dos dados de entrada. Em contraste, a interação entre algoritmo e estresse não foi estatisticamente significativa (F = 1,84; p = 0,089; η²p = 0,014), indicando que os algoritmos apresentaram comportamento semelhante frente aos diferentes cenários de contenção de recursos.